# MC Sim - GPU Build & Test

Build and test the molecular communication GPU simulator on Colab.

**Before running:** Go to Runtime > Change runtime type > T4 GPU

In [ ]:
# Verify GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
# Clone the repo
!git clone https://github.com/alwaysEpic/molecular_modeling_gpu.git
%cd molecular_modeling_gpu

In [ ]:
# Build both targets + RNG dump tool
!mkdir -p build && cd build && cmake .. && make -j$(nproc)
!g++ -O2 -o build/dump_rng_cpu scripts/dump_rng_cpu.cpp -lm
!pip install -q numpy matplotlib scipy
!ls -la build/mc_sim build/mc_sim_cpu build/dump_rng_cpu

## 1. GPU Smoke Test

In [ ]:
!cd build && ./mc_sim -i 1000 -f -v

## 2. CPU vs GPU Comparison

In [ ]:
# 1000 paths
!cd build && ./mc_sim -i 1000 -c -f -v

In [ ]:
# 10000 paths
!cd build && ./mc_sim -i 10000 -c -f -v

## 3. CPU-Only Benchmarks

In [ ]:
!cd build && ./mc_sim_cpu -i 1000 -f -v
print("---")
!cd build && ./mc_sim_cpu -i 1000 -f -v -n

## 4. Analytical Validation — 1D First-Hit with Drift

Compares simulation output against analytical inverse Gaussian (thesis eq 4.3).

In [ ]:
# GPU validation
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_d_wide.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2

In [ ]:
# CPU validation
!cd build && ./mc_sim_cpu -i 10000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_h.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2

## 5. Analytical Validation — 3D Spherical Receiver (no drift)

Compares against analytical H_Diff (thesis eq 4.1).

In [ ]:
# GPU - 3D diffusion only
!cd build && ./mc_sim -i 10000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_d_wide.csv \
    --no-plot --total-paths 10000

In [ ]:
# CPU - 3D diffusion only
!cd build && ./mc_sim_cpu -i 10000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_h.csv \
    --no-plot --total-paths 10000

## 6. CPU/GPU Agreement

KS test comparing hit-time distributions from both backends.

In [ ]:
# Run both backends with same params (different RNG, so not identical)
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_d_wide.csv \
    --timestep 1E-7

## 7. RNG Quality

Tests statistical moments, autocorrelation, and normality.

In [ ]:
# CPU RNG quality
!cd build && ./dump_rng_cpu 10000 > rng_cpu.csv
!python scripts/validate_rng.py build/rng_cpu.csv --no-plot

In [ ]:
%%writefile /tmp/dump_rng_gpu.cu
// Dump GPU random numbers for RNG quality testing
#include <stdio.h>
#include <stdlib.h>
#include <curand.h>
#include <curand_kernel.h>

__global__ void gen_randn(curandStatePhilox4_32_10_t* rng, float* out, int n, int draws_per_thread) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  for (int d = 0; d < draws_per_thread; d++) {
    out[idx * draws_per_thread + d] = curand_normal(&rng[idx]);
  }
}

__global__ void init_rng(curandStatePhilox4_32_10_t* rng, long long seed, int n) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  curand_init(seed, idx, 0, &rng[idx]);
}

int main(int argc, char** argv) {
  int n_threads = 1000;
  int draws = 10;  // 10 draws per thread = 10000 total
  int total = n_threads * draws;
  long long seed = 42;
  if (argc > 1) seed = atoll(argv[1]);

  curandStatePhilox4_32_10_t* d_rng;
  float* d_out;
  cudaMalloc(&d_rng, n_threads * sizeof(curandStatePhilox4_32_10_t));
  cudaMalloc(&d_out, total * sizeof(float));

  int block = 128;
  int grid = (n_threads + block - 1) / block;
  init_rng<<<grid, block>>>(d_rng, seed, n_threads);
  cudaDeviceSynchronize();
  gen_randn<<<grid, block>>>(d_rng, d_out, n_threads, draws);
  cudaDeviceSynchronize();

  float* out = (float*)malloc(total * sizeof(float));
  cudaMemcpy(out, d_out, total * sizeof(float), cudaMemcpyDeviceToHost);

  for (int i = 0; i < total; i++) printf("%0.15f\n", out[i]);

  free(out);
  cudaFree(d_rng);
  cudaFree(d_out);
  return 0;
}

In [ ]:
# Compile and run GPU RNG dump, then validate
!nvcc -O2 -o build/dump_rng_gpu /tmp/dump_rng_gpu.cu
!cd build && ./dump_rng_gpu 42 > rng_gpu.csv
print("=== GPU RNG (persistent Philox, seed=42) ===")
!python scripts/validate_rng.py build/rng_gpu.csv --no-plot
print()
!cd build && ./dump_rng_gpu 123 > rng_gpu2.csv
print("=== GPU RNG (persistent Philox, seed=123) ===")
!python scripts/validate_rng.py build/rng_gpu2.csv --no-plot

## 8. Reproducibility (--seed)

Same seed should produce identical output.

In [ ]:
# GPU reproducibility
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_d_wide.csv run1_gpu.csv
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_d_wide.csv run2_gpu.csv
!diff build/run1_gpu.csv build/run2_gpu.csv && echo 'GPU REPRODUCIBILITY: PASS (identical)' || echo 'GPU REPRODUCIBILITY: FAIL (differs)'

In [ ]:
# CPU reproducibility
!cd build && ./mc_sim_cpu -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_h.csv run1_cpu.csv
!cd build && ./mc_sim_cpu -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_h.csv run2_cpu.csv
!diff build/run1_cpu.csv build/run2_cpu.csv && echo 'CPU REPRODUCIBILITY: PASS (identical)' || echo 'CPU REPRODUCIBILITY: FAIL (differs)'